# Introduction to Data Analysis with Python and Pandas

**Estimated time:** 2–3 hours (including exercises)

Welcome! In this lesson you'll learn how to analyze real-world data using **pandas**, Python's most widely used data analysis library. By the end, you will be able to take a raw dataset and turn it into concrete answers and charts.

## What you'll learn

1. What data analysis is and how pandas fits in
2. The two core pandas structures: **Series** and **DataFrame**
3. Loading data and getting a first impression of it
4. Selecting and filtering data
5. Sorting and creating new columns
6. Dealing with missing values
7. Grouping and aggregating (`groupby`) — where analysis gets powerful
8. Creating basic visualizations
9. A mini-project that puts it all together

## Prerequisites

You should be comfortable with core Python: variables, lists, dictionaries, functions, and `for` loops. No prior pandas experience is needed.

## How to use this notebook

- Run every code cell as you read (`Shift + Enter`). Don't just read — **run and experiment**.
- Each section ends with short **✏️ Exercises**. Do them before moving on; solutions are at the very end.
- 💡 boxes contain tips; ⚠️ boxes contain common mistakes to avoid.

---

## 1. What is data analysis?

Data analysis is the process of turning raw data into **insight**: cleaning it, exploring it, summarizing it, and communicating what it means. A typical workflow looks like this:

**Question → Get data → Clean data → Explore & summarize → Visualize → Communicate**

### Why pandas?

Plain Python *can* process data with lists and dictionaries, but it quickly becomes slow and verbose. Pandas gives you:

- A **table structure** (the DataFrame) that works like a supercharged spreadsheet
- One-line operations for filtering, grouping, and summarizing millions of rows
- Built-in handling for messy realities: missing values, mixed types, dates
- Easy reading/writing of CSV, Excel, JSON, SQL and more

Pandas is built on top of **NumPy**, a fast numerical computing library, which is why it can crunch large datasets far faster than plain Python loops.

---

## 2. Setup

Pandas is not part of Python's standard library. If you're using **Google Colab** or **Anaconda**, it's already installed. Otherwise, install it once from your terminal:

```bash
pip install pandas matplotlib
```

Then import it. The alias `pd` is a universal convention — every pandas tutorial and codebase you'll ever read uses it.

In [2]:
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)

pandas version: 3.0.5


💡 **Tip:** If the cell above raises `ModuleNotFoundError`, pandas isn't installed in the Python environment your notebook is using. Run `pip install pandas` and restart the kernel.

---

## 3. Our dataset: coffee shop sales ☕

Throughout this lesson we'll analyze sales data from a small (fictional) chain of coffee shops with three branches. Each row is one sale.

The cell below **generates the dataset and saves it as `coffee_sales.csv`**, so this notebook is fully self-contained. You don't need to understand this generation code — in real life the CSV would simply be given to you. Just run it.

In [3]:
# Run this cell to create the dataset file. You don't need to study this code.
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)  # seed makes the "random" data reproducible
n = 400

products = {
    "Espresso": ("Coffee", 2.50), "Latte": ("Coffee", 4.00),
    "Cappuccino": ("Coffee", 3.80), "Green Tea": ("Tea", 3.00),
    "Chai Latte": ("Tea", 4.20), "Croissant": ("Food", 3.50),
    "Sandwich": ("Food", 6.50), "Muffin": ("Food", 3.20),
}
names = list(products.keys())
chosen = rng.choice(names, size=n, p=[0.18, 0.22, 0.15, 0.08, 0.07, 0.12, 0.08, 0.10])

data = pd.DataFrame({
    "date": rng.choice(pd.date_range("2026-03-01", "2026-03-31"), size=n),
    "branch": rng.choice(["Downtown", "Airport", "University"], size=n, p=[0.45, 0.25, 0.30]),
    "product": chosen,
    "category": [products[p][0] for p in chosen],
    "quantity": rng.integers(1, 5, size=n),
    "unit_price": [products[p][1] for p in chosen],
    "customer_rating": np.round(rng.uniform(2.5, 5.0, size=n), 1),
})

# Real data is messy: knock out some ratings to create missing values
missing_idx = rng.choice(n, size=48, replace=False)
data.loc[missing_idx, "customer_rating"] = np.nan

data = data.sort_values("date").reset_index(drop=True)
data.to_csv("coffee_sales.csv", index=False)
print("Created coffee_sales.csv with", len(data), "rows")

Created coffee_sales.csv with 400 rows


---

## 4. The building blocks: Series and DataFrame

Pandas has two core data structures:

| Structure | What it is | Spreadsheet analogy |
|---|---|---|
| **Series** | A 1-dimensional labeled array | A single column |
| **DataFrame** | A 2-dimensional table of Series | A whole sheet |

### 4.1 Series

A Series is like a list, but every value has a **label** (its *index*), and all values share one data type.

In [3]:
prices = pd.Series([2.50, 4.00, 3.80], index=["Espresso", "Latte", "Cappuccino"])
prices

Espresso      2.5
Latte         4.0
Cappuccino    3.8
dtype: float64

In [4]:
price=pd.Series([1,2,3,4], index=["Coke", "Fanta", "Apple", "Mango"])
price

Coke     1
Fanta    2
Apple    3
Mango    4
dtype: int64

In [5]:
print(price["Coke"])
print(price.iloc[2])
print(price.mean())

1
3
2.5


The left side is the **index** (labels), the right side is the values, and `dtype: float64` tells you the data type. You can access values by label or position:

In [6]:
print(prices["Latte"])       # access by label
print(prices.iloc[0])         # access by position (first element)
print(prices.mean())          # Series come with built-in statistics

4.0
2.5
3.4333333333333336


### 4.2 DataFrame

A DataFrame is a collection of Series that share the same row index — in other words, a table. You can build one from a dictionary, where keys become column names:

In [7]:
mini = pd.DataFrame({
    "product": ["Espresso", "Latte", "Cappuccino"],
    "price": [2.50, 4.00, 3.80],
    "in_stock": [True, True, False],
})
mini

,product,price,in_stock
0,Espresso,2.5,True
1,Latte,4.0,True
2,Cappuccino,3.8,False


In [8]:
football=pd.DataFrame({
    "Goalkeeper": ["Kepa","Martinez", "Sanchez"],
    "Defenders":["Lacroix", "Ramos", "Maldini"],
})
football

,Goalkeeper,Defenders
0,Kepa,Lacroix
1,Martinez,Ramos
2,Sanchez,Maldini


Notice the bold numbers on the left: that's the row index (0, 1, 2 by default). Each column is a Series, and different columns can have different dtypes (here: text, float, boolean).

⚠️ **Common mistake:** building a DataFrame from a dict where the lists have different lengths raises `ValueError: All arrays must be of the same length`. Every column needs one value per row.

### ✏️ Exercises 4

**4.1** Create a Series called `temps` holding the values `18.5, 21.0, 19.5, 23.0` with the index `["Mon", "Tue", "Wed", "Thu"]`. Print the temperature for `"Wed"` and the maximum temperature (hint: `.max()`).

**4.2** Create a DataFrame called `menu` with columns `item` (three drink names of your choice), `size_ml` (three integers), and `price` (three floats).

In [19]:
# Your solution for exercises 4.1 and 4.2 here
temps=pd.Series([18.5,21.0,19.5,23.0], index=["Mon", "Tue", "Wed", "Thu"])
print(temps["Wed"])
print(temps.max())

menu=pd.DataFrame({
    "items": ["Biscuit", "Doritos", "Simba"],
    "size_ml": [500, 600, 700],
    "price": [12, 22, 19]
})
menu

19.5
23.0


,items,size_ml,price
0,Biscuit,500,12
1,Doritos,600,22
2,Simba,700,19


---

## 5. Loading data and taking a first look

In practice you rarely type data by hand — you load it from a file. The workhorse is `pd.read_csv()`:

In [9]:
df=pd.read_csv("coffee_sales.csv")
df.columns

Index(['date', 'branch', 'product', 'category', 'quantity', 'unit_price',
       'customer_rating'],
      dtype='str')

In [10]:
df = pd.read_csv("coffee_sales.csv", parse_dates=["date"])
df.head()

,date,branch,product,category,quantity,unit_price,customer_rating
0,2026-03-01,Airport,Croissant,Food,4,3.5,3.9
1,2026-03-01,University,Latte,Coffee,4,4.0,2.9
2,2026-03-01,University,Croissant,Food,4,3.5,3.2
3,2026-03-01,Downtown,Muffin,Food,1,3.2,3.2
4,2026-03-01,University,Chai Latte,Tea,2,4.2,2.6


`parse_dates=["date"]` tells pandas to treat that column as real dates instead of plain text — essential if you ever want to sort or filter by date.

💡 Pandas can also read other formats: `pd.read_excel()`, `pd.read_json()`, `pd.read_sql()`, and more.

### 5.1 The first-look routine

Whenever you load a new dataset, run this four-step inspection **before doing anything else**. It takes 30 seconds and tells you what you're dealing with.

In [11]:
df.shape  # (number of rows, number of columns)

(400, 7)

In [12]:
df.head()      # first 5 rows -- also try df.tail() for the last 5
df.sample(5)   # 5 random rows (often more representative than the first 5)

,date,branch,product,category,quantity,unit_price,customer_rating
295,2026-03-23,Downtown,Latte,Coffee,4,4.0,2.9
187,2026-03-15,University,Chai Latte,Tea,1,4.2,3.8
260,2026-03-20,University,Croissant,Food,1,3.5,2.9
381,2026-03-30,University,Green Tea,Tea,4,3.0,3.7
37,2026-03-03,University,Cappuccino,Coffee,1,3.8,3.8


In [13]:
df.info()  # column names, dtypes, and how many non-missing values each has

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             400 non-null    datetime64[us]
 1   branch           400 non-null    str           
 2   product          400 non-null    str           
 3   category         400 non-null    str           
 4   quantity         400 non-null    int64         
 5   unit_price       400 non-null    float64       
 6   customer_rating  352 non-null    float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 22.0 KB


Read `df.info()` carefully — it's packed with information:

- **RangeIndex: 400 entries** → 400 rows
- Each column's **Non-Null Count** → `customer_rating` has only 352 non-null values out of 400, so **48 are missing**. We'll deal with that in section 8.
- **Dtype** → `str` or `object` means text, `int64`/`float64` are numbers, `datetime64` is a date. (Older pandas versions show text as `object`; pandas 3.x shows `str` — they mean the same thing here.)

⚠️ **Common mistake:** a column that *should* be numbers but shows dtype `object`/`str` means pandas read it as *text* (often caused by stray characters like `"$4.00"` or `"N/A"`). Math on it will fail or misbehave until you convert it with `pd.to_numeric()`.

In [14]:
df.describe()  # summary statistics for the numeric columns

,date,quantity,unit_price,customer_rating
count,400,400.000000,400.000000,352.000000
mean,2026-03-15 22:55:12,2.500000,3.649750,3.746591
min,2026-03-01 00:00:00,1.000000,2.500000,2.500000
25%,2026-03-08 00:00:00,2.000000,3.000000,3.100000
50%,2026-03-16 00:00:00,2.000000,3.500000,3.700000
75%,2026-03-24 00:00:00,3.000000,4.000000,4.400000
max,2026-03-31 00:00:00,4.000000,6.500000,5.000000
std,NaN,1.096817,0.995478,0.730913


From one line we already learn a lot: quantities range from 1 to 4, the average rating is around 3.7, and prices range from 2.50 to 6.50.

For text columns, two other methods shine:

In [15]:
print(df["branch"].unique())          # the distinct values
df["product"].value_counts()           # how often each value occurs

<StringArray>
['Airport', 'University', 'Downtown']
Length: 3, dtype: str


product
Espresso      82
Latte         76
Cappuccino    58
Croissant     51
Muffin        36
Chai Latte    34
Green Tea     33
Sandwich      30
Name: count, dtype: int64

### ✏️ Exercises 5

**5.1** How many rows and columns does `df` have? Get the answer with code, not by counting.

**5.2** Use `value_counts()` to find which **category** (not product) appears most often.

**5.3** What is the highest `customer_rating` in the dataset? (Hint: select the column, then use a method you saw in section 4.)

In [16]:
# Your solutions for exercises 5 here
df.shape
df["category"].value_counts()
df["customer_rating"].max()

np.float64(5.0)

---

## 6. Selecting and filtering data

Analysis almost always starts with narrowing down: *these* columns, *those* rows.

### 6.1 Selecting columns

In [17]:
df["product"].head()  # single column -> returns a Series

0     Croissant
1         Latte
2     Croissant
3        Muffin
4    Chai Latte
Name: product, dtype: str

In [18]:
df[["product", "quantity", "unit_price"]].head()  # multiple columns -> DataFrame
# note the DOUBLE brackets: you're passing a LIST of column names

,product,quantity,unit_price
0,Croissant,4,3.5
1,Latte,4,4.0
2,Croissant,4,3.5
3,Muffin,1,3.2
4,Chai Latte,2,4.2


⚠️ **Common mistake:** `df["product", "quantity"]` (single brackets, two names) raises a `KeyError`. Pandas thinks you're asking for one column whose name is the tuple `("product", "quantity")`. Use double brackets: `df[["product", "quantity"]]`.

### 6.2 Filtering rows with boolean conditions

This is one of the most important patterns in pandas. A comparison on a column produces a Series of `True`/`False` values — a **boolean mask** — and putting that mask inside `df[...]` keeps only the `True` rows.

In [19]:
mask = df["quantity"] >= 3
mask.head()  # look at what a mask actually is: a Series of True/False

0     True
1     True
2     True
3    False
4    False
Name: quantity, dtype: bool

In [20]:
big_orders = df[df["quantity"] >= 3]
print(len(big_orders), "of", len(df), "sales had quantity 3 or more")
big_orders.head()

196 of 400 sales had quantity 3 or more


,date,branch,product,category,quantity,unit_price,customer_rating
0,2026-03-01,Airport,Croissant,Food,4,3.5,3.9
1,2026-03-01,University,Latte,Coffee,4,4.0,2.9
2,2026-03-01,University,Croissant,Food,4,3.5,3.2
7,2026-03-01,Downtown,Espresso,Coffee,3,2.5,2.8
9,2026-03-01,Downtown,Muffin,Food,4,3.2,NaN


Combine conditions with `&` (and), `|` (or), and `~` (not). **Each condition must be wrapped in parentheses.**

In [21]:
# Coffee sales at the Downtown branch
downtown_coffee = df[(df["branch"] == "Downtown") & (df["category"] == "Coffee")]
print(len(downtown_coffee), "rows")
downtown_coffee.head(3)

94 rows


,date,branch,product,category,quantity,unit_price,customer_rating
7,2026-03-01,Downtown,Espresso,Coffee,3,2.5,2.8
10,2026-03-01,Downtown,Espresso,Coffee,4,2.5,2.9
17,2026-03-02,Downtown,Latte,Coffee,4,4.0,4.3


In [22]:
men=df[(df["product"] == "Espresso") & (df["quantity"]>=1)]
men.head()

,date,branch,product,category,quantity,unit_price,customer_rating
6,2026-03-01,University,Espresso,Coffee,2,2.5,3.6
7,2026-03-01,Downtown,Espresso,Coffee,3,2.5,2.8
10,2026-03-01,Downtown,Espresso,Coffee,4,2.5,2.9
22,2026-03-02,Downtown,Espresso,Coffee,3,2.5,4.4
25,2026-03-02,Airport,Espresso,Coffee,4,2.5,NaN


⚠️ **Common mistakes when filtering:**
- Using Python's `and`/`or` instead of `&`/`|` → `ValueError: The truth value of a Series is ambiguous`.
- Forgetting parentheses: `df[df["a"] > 1 & df["b"] < 2]` fails because `&` binds tighter than `>`. Write `df[(df["a"] > 1) & (df["b"] < 2)]`.
- Using `=` (assignment) instead of `==` (comparison) inside a condition.

💡 For matching several values, `.isin()` beats chaining `|`:

In [23]:
hot_drinks = df[df["product"].isin(["Espresso", "Latte", "Cappuccino"])]
hot_drinks["product"].value_counts()

product
Espresso      82
Latte         76
Cappuccino    58
Name: count, dtype: int64

### 6.3 `.loc` and `.iloc`: precise selection

Square brackets alone are convenient but limited. The two precision tools are:

- **`.loc[rows, columns]`** — select by **labels** (and boolean masks)
- **`.iloc[rows, columns]`** — select by **integer positions**

In [24]:
df.loc[0, "product"]        # row with index label 0, column "product" 

'Croissant'

In [25]:
df.loc[df["category"] == "Tea", ["product", "quantity", "unit_price"]].head()
# rows: where category is Tea | columns: just these three

,product,quantity,unit_price
4,Chai Latte,2,4.2
19,Green Tea,3,3.0
20,Green Tea,3,3.0
27,Chai Latte,3,4.2
28,Green Tea,2,3.0


In [26]:
df.iloc[0:3, 0:4]  # first 3 rows, first 4 columns, purely by position

,date,branch,product,category
0,2026-03-01,Airport,Croissant,Food
1,2026-03-01,University,Latte,Coffee
2,2026-03-01,University,Croissant,Food


💡 **Rule of thumb:** use `.loc` with a boolean mask when you want *specific rows and specific columns at once*. Use `.iloc` when position genuinely matters (e.g., "the last 10 rows": `df.iloc[-10:]`).

### ✏️ Exercises 6

**6.1** Select all sales at the **Airport** branch. How many are there?

**6.2** Select all **Food** sales with a `customer_rating` of at least 4.5. Show only the `product`, `quantity`, and `customer_rating` columns (use `.loc`).

**6.3** Using `.iloc`, show the last 5 rows and the first 3 columns of `df`.

In [27]:
# Your solutions for exercises 6 here


df.iloc[-5:,:3]

,date,branch,product
395,2026-03-31,Airport,Latte
396,2026-03-31,Airport,Croissant
397,2026-03-31,Downtown,Latte
398,2026-03-31,Downtown,Chai Latte
399,2026-03-31,Downtown,Latte


---

## 7. Sorting and creating new columns

### 7.1 Sorting

In [28]:
df.sort_values("customer_rating", ascending=False).head()  # best-rated sales first

,date,branch,product,category,quantity,unit_price,customer_rating
234,2026-03-19,Airport,Muffin,Food,3,3.2,5.0
265,2026-03-21,Airport,Latte,Coffee,3,4.0,5.0
108,2026-03-09,Airport,Latte,Coffee,4,4.0,5.0
79,2026-03-06,Downtown,Cappuccino,Coffee,1,3.8,5.0
314,2026-03-25,Downtown,Espresso,Coffee,2,2.5,5.0


In [30]:
df.sort_values("quantity").head() #When you want to sort a single column in ascending order, there is no reason to write ascending=True, you can only do that if you are sorting out 2 columns at the same time

,date,branch,product,category,quantity,unit_price,customer_rating
369,2026-03-29,Downtown,Chai Latte,Tea,1,4.2,3.6
371,2026-03-29,Downtown,Sandwich,Food,1,6.5,3.7
30,2026-03-02,Downtown,Croissant,Food,1,3.5,3.4
358,2026-03-28,University,Muffin,Food,1,3.2,3.1
357,2026-03-28,University,Cappuccino,Coffee,1,3.8,3.0


In [48]:
# Sort by several columns: branch A->Z, then quantity high->low within each branch
df.sort_values(["branch", "quantity"], ascending=[True, False]).head()

,date,branch,product,category,quantity,unit_price,customer_rating
0,2026-03-01,Airport,Croissant,Food,4,3.5,3.9
25,2026-03-02,Airport,Espresso,Coffee,4,2.5,NaN
26,2026-03-02,Airport,Muffin,Food,4,3.2,3.4
31,2026-03-03,Airport,Espresso,Coffee,4,2.5,2.7
41,2026-03-03,Airport,Espresso,Coffee,4,2.5,4.3


💡 `sort_values()` returns a **new, sorted copy** — it does not change `df` itself. If you want to keep the sorted version, assign it: `df = df.sort_values(...)`. Most pandas operations work this way.

### 7.2 Creating new columns

Assign to a column name that doesn't exist yet, and pandas creates it. Operations are **vectorized**: writing `df["a"] * df["b"]` multiplies the columns row by row — no loop needed.

In [31]:
df["revenue"] = df["quantity"] * df["unit_price"]
df[["product", "quantity", "unit_price", "revenue"]].head()

,product,quantity,unit_price,revenue
0,Croissant,4,3.5,14.0
1,Latte,4,4.0,16.0
2,Croissant,4,3.5,14.0
3,Muffin,1,3.2,3.2
4,Chai Latte,2,4.2,8.4


In [35]:
print("Total revenue for March: $", df["revenue"].sum().round(2))

Total revenue for March: $ 3636.8


You can also derive columns from dates — very common in real analysis:

In [36]:
df["weekday"] = df["date"].dt.day_name()   # .dt gives access to date parts
df[["date", "weekday", "product", "revenue"]].head(3)

,date,weekday,product,revenue
0,2026-03-01,Sunday,Croissant,14.0
1,2026-03-01,Sunday,Latte,16.0
2,2026-03-01,Sunday,Croissant,14.0


### ✏️ Exercises 7

**7.1** Sort `df` by `revenue` (highest first) and display the top 5 sales.

**7.2** Create a column `is_large_order` that is `True` when `quantity` is 3 or more. (Hint: a comparison already produces True/False — just assign it.)

**7.3** What is the *average* revenue per sale? Round it to 2 decimals.

In [ ]:
# Your solutions for exercises 7 here
df.sort_values("revenue", ascending=False).head()
df["is_large_order"]=df["quantity"]>=3
df.head()


,date,branch,product,category,quantity,unit_price,customer_rating,revenue,weekday,is_large_order
0,2026-03-01,Airport,Croissant,Food,4,3.5,3.9,14.0,Sunday,True
1,2026-03-01,University,Latte,Coffee,4,4.0,2.9,16.0,Sunday,True
2,2026-03-01,University,Croissant,Food,4,3.5,3.2,14.0,Sunday,True
3,2026-03-01,Downtown,Muffin,Food,1,3.2,3.2,3.2,Sunday,False
4,2026-03-01,University,Chai Latte,Tea,2,4.2,2.6,8.4,Sunday,False


---

## 8. Handling missing values

Real datasets are almost never complete. Pandas marks missing values as `NaN` ("Not a Number"). Our `customer_rating` column has them — customers don't always leave a rating.

### 8.1 Detecting

In [ ]:
df.isna().sum()  # count missing values per column

If you want to *look* at the incomplete rows:

In [ ]:
df[df["customer_rating"].isna()].head(3)

### 8.2 Three ways to deal with missing data

| Strategy | Code | When to use |
|---|---|---|
| **Drop** the rows | `df.dropna()` | Few rows affected, and they carry little information |
| **Fill** with a value | `df.fillna(value)` | You have a sensible default or estimate (e.g., the mean) |
| **Leave them** | — | Many pandas functions (like `.mean()`) skip NaN automatically |

⚠️ There is no universally correct choice — it depends on *why* the data is missing and what you'll do with it. Filling ratings with the mean, for example, quietly makes your data look more average than it really is. Always be able to justify your choice.

In [ ]:
print("Rows before dropna:", len(df))
print("Rows after dropna: ", len(df.dropna()))   # returns a copy; df is unchanged

In [ ]:
# Filling: create a new column instead of overwriting the original -- good practice
mean_rating = df["customer_rating"].mean()   # .mean() skips NaN automatically
df["rating_filled"] = df["customer_rating"].fillna(round(mean_rating, 1))
df[["customer_rating", "rating_filled"]].head(6)

💡 `dropna()` and `fillna()` return **copies** by default, like `sort_values()`. Your original DataFrame is safe unless you reassign it.

### ✏️ Exercises 8

**8.1** How many values are missing in the whole DataFrame? (Hint: you can chain `.sum()` twice.)

**8.2** Create a DataFrame `rated` that contains only rows where `customer_rating` is **not** missing. Verify with `.isna().sum()` that it has no missing ratings. (Hint: `.notna()` is the opposite of `.isna()`.)

In [ ]:
# Your solutions for exercises 8 here


---

## 9. Grouping and aggregating: `groupby`

So far we've summarized *whole columns*. The real power move is summarizing **per group**: revenue *per branch*, average rating *per product*, sales *per weekday*. That's `groupby` — arguably the most important tool in pandas.

The mental model is **split → apply → combine**:

1. **Split** the rows into groups (e.g., one group per branch)
2. **Apply** an aggregation to each group (e.g., sum the revenue)
3. **Combine** the results into a new table

In [ ]:
df.groupby("branch")["revenue"].sum()

Read it right to left: *take revenue, per branch, and sum it.* One line answers a genuine business question.

In [ ]:
df.groupby("product")["customer_rating"].mean().sort_values(ascending=False)

In [ ]:
# .agg() applies several aggregations at once
df.groupby("branch")["revenue"].agg(["count", "sum", "mean"]).round(2)

In [ ]:
# Group by TWO columns: revenue per category within each branch
df.groupby(["branch", "category"])["revenue"].sum().round(2)

💡 Useful aggregation functions: `sum`, `mean`, `median`, `min`, `max`, `count`, `nunique` (number of distinct values), `std` (standard deviation).

⚠️ **Common mistake:** forgetting to select a column and/or an aggregation. `df.groupby("branch")` alone just returns a `GroupBy` object — you must follow it with something like `["revenue"].sum()`.

### ✏️ Exercises 9

**9.1** What is the **total quantity** of items sold per **category**?

**9.2** What is the **average revenue** per **weekday**? Sort the result from highest to lowest. Which day is best for business?

**9.3** For each **product**, how many sales does it have and what is its total revenue? (Hint: `.agg(["count", "sum"])` on the revenue column.)

In [ ]:
# Your solutions for exercises 9 here


---

## 10. Basic visualization

Numbers convince analysts; pictures convince everyone. Pandas has built-in plotting (using the **matplotlib** library under the hood): call `.plot()` on a Series or DataFrame.

In [ ]:
import matplotlib.pyplot as plt  # standard alias, used to fine-tune plots

df.groupby("branch")["revenue"].sum().plot(kind="bar", title="Total revenue by branch")
plt.ylabel("Revenue ($)")
plt.show()

The pattern to remember: **aggregate first, then plot the result.** A groupby result is a Series, and Series know how to draw themselves.

Common plot kinds:

| `kind=` | Best for |
|---|---|
| `"bar"` | Comparing categories |
| `"line"` | Trends over time |
| `"hist"` | The distribution of one numeric column |
| `"scatter"` | Relationship between two numeric columns |

In [ ]:
# A line plot: daily revenue over the month
df.groupby("date")["revenue"].sum().plot(kind="line", title="Daily revenue in March")
plt.ylabel("Revenue ($)")
plt.show()

In [ ]:
# A histogram: how are customer ratings distributed?
df["customer_rating"].plot(kind="hist", bins=10, title="Distribution of customer ratings")
plt.xlabel("Rating")
plt.show()

### ✏️ Exercises 10

**10.1** Make a **bar chart** of the number of sales per product. (Hint: `value_counts()` produces exactly the Series you need.)

**10.2** Make a **bar chart** of average revenue per weekday, using your groupby from exercise 9.2. Give it a title and a y-axis label.

In [ ]:
# Your solutions for exercises 10 here


---

## 11. Mini-project: advise the owner 🎯

Time to put everything together — no new concepts, just you and the data. The owner of the coffee chain asks you four questions. Answer each one with code, and write your conclusion in a sentence (add a markdown cell or a `print()`).

**Q1.** Which single **product** generates the most total revenue?

**Q2.** The owner suspects the **Airport** branch sells more food (relative to drinks) than other branches. Compute revenue per category *within each branch* and see if the data supports this.

**Q3.** The owner is considering closing on the **weekend**. What share of total revenue comes from Saturday and Sunday? (Hint: filter with `.isin(["Saturday", "Sunday"])`, then compare sums.)

**Q4.** Is there any product with a worryingly low **average rating** (below 3.5)? Should the owner investigate?

Make one **chart** to support your most interesting finding.

In [ ]:
# Q1: your code here


In [ ]:
# Q2: your code here


In [ ]:
# Q3: your code here


In [ ]:
# Q4 + supporting chart: your code here


---

## 12. Common errors survival guide 🚑

Bookmark this section — you *will* meet these.

| Error message | Usual cause | Fix |
|---|---|---|
| `KeyError: 'colname'` | Typo in a column name, wrong capitalization, or hidden spaces | Check `df.columns`; use `df.columns = df.columns.str.strip()` for stray spaces |
| `ModuleNotFoundError: No module named 'pandas'` | Pandas not installed in this environment | `pip install pandas`, restart kernel |
| `FileNotFoundError` on `read_csv` | Wrong filename or the file isn't in the notebook's folder | Check the path; try the full path |
| `ValueError: The truth value of a Series is ambiguous` | Used `and`/`or`/`if` on a whole column | Use `&` and `\|` with parentheses for masks |
| `KeyError` with double column selection | `df["a", "b"]` instead of `df[["a", "b"]]` | Double brackets for multiple columns |
| `SettingWithCopyWarning` / `ChainedAssignmentError` | Assigning via chained selection: `df[df.a > 1]["b"] = 0` | Use one `.loc`: `df.loc[df.a > 1, "b"] = 0` |
| `TypeError` when doing math on a column | Column dtype is `object`/`str` (text) | Convert: `pd.to_numeric(df["col"])` |
| Dates won't sort/filter correctly | Date column read as text | `parse_dates=[...]` in `read_csv`, or `pd.to_datetime()` |

💡 **General debugging routine:** print `df.columns` and `df.dtypes`, look at `df.head()`, and read the *last line* of the error message first.

---

## 13. One-page cheat sheet 📋

```python
import pandas as pd

# LOAD & INSPECT
df = pd.read_csv("file.csv", parse_dates=["date"])
df.shape; df.head(); df.info(); df.describe()
df["col"].unique(); df["col"].value_counts()

# SELECT
df["col"]                          # one column (Series)
df[["a", "b"]]                     # several columns (DataFrame)
df.loc[mask, ["a", "b"]]           # rows by condition, chosen columns
df.iloc[0:5, 0:3]                  # rows/columns by position

# FILTER
df[df["col"] > 10]
df[(df["a"] > 1) & (df["b"] == "x")]      # & and | need parentheses
df[df["col"].isin(["x", "y"])]

# SORT & NEW COLUMNS
df.sort_values("col", ascending=False)
df["new"] = df["a"] * df["b"]
df["weekday"] = df["date"].dt.day_name()

# MISSING VALUES
df.isna().sum()
df.dropna()                        # returns a copy
df["col"].fillna(value)            # returns a copy

# GROUP & AGGREGATE  (split -> apply -> combine)
df.groupby("key")["val"].sum()
df.groupby("key")["val"].agg(["count", "mean", "sum"])
df.groupby(["k1", "k2"])["val"].mean()

# PLOT (aggregate first, then plot)
df.groupby("key")["val"].sum().plot(kind="bar")   # bar, line, hist, scatter

# SAVE
df.to_csv("out.csv", index=False)
```

### Where to go next

- **Official user guide & tutorials:** pandas.pydata.org/docs — start with "10 minutes to pandas"
- Next topics in a natural order: merging/joining tables (`pd.merge`), pivot tables, string methods (`.str`), time series resampling
- Practice beats reading: find any CSV that interests you (sports, music, weather) and repeat this lesson's workflow on it

---

## 14. Solutions ✅

Try the exercises honestly before peeking!

### Exercises 4

In [ ]:
# 4.1
temps = pd.Series([18.5, 21.0, 19.5, 23.0], index=["Mon", "Tue", "Wed", "Thu"])
print(temps["Wed"], temps.max())

# 4.2
menu = pd.DataFrame({
    "item": ["Latte", "Mocha", "Flat White"],
    "size_ml": [300, 350, 250],
    "price": [4.00, 4.50, 3.90],
})
menu

### Exercises 5

In [ ]:
# 5.1
print(df.shape)               # (rows, columns) -- note df now has extra columns we created

# 5.2
print(df["category"].value_counts())   # Coffee appears most often

# 5.3
print(df["customer_rating"].max())

### Exercises 6

In [ ]:
# 6.1
airport = df[df["branch"] == "Airport"]
print(len(airport), "airport sales")

# 6.2
food_top = df.loc[(df["category"] == "Food") & (df["customer_rating"] >= 4.5),
                  ["product", "quantity", "customer_rating"]]
print(food_top.head())

# 6.3
df.iloc[-5:, 0:3]

### Exercises 7

In [ ]:
# 7.1
print(df.sort_values("revenue", ascending=False).head())

# 7.2
df["is_large_order"] = df["quantity"] >= 3
print(df["is_large_order"].value_counts())

# 7.3
print(round(df["revenue"].mean(), 2))

### Exercises 8

In [ ]:
# 8.1
print(df.isna().sum().sum())   # per-column counts, then their total

# 8.2
rated = df[df["customer_rating"].notna()]
print(rated.isna().sum())

### Exercises 9

In [ ]:
# 9.1
print(df.groupby("category")["quantity"].sum())

# 9.2
print(df.groupby("weekday")["revenue"].mean().sort_values(ascending=False))

# 9.3
print(df.groupby("product")["revenue"].agg(["count", "sum"]).round(2))

### Exercises 10

In [ ]:
# 10.1
df["product"].value_counts().plot(kind="bar", title="Number of sales per product")
plt.ylabel("Sales")
plt.show()

# 10.2
df.groupby("weekday")["revenue"].mean().sort_values(ascending=False).plot(
    kind="bar", title="Average revenue per weekday")
plt.ylabel("Average revenue ($)")
plt.show()

### Mini-project (one possible solution)

In [ ]:
# Q1: product with most total revenue
rev_by_product = df.groupby("product")["revenue"].sum().sort_values(ascending=False)
print(rev_by_product)
print("Top earner:", rev_by_product.index[0])

In [ ]:
# Q2: revenue per category within each branch
by_branch_cat = df.groupby(["branch", "category"])["revenue"].sum().round(2)
print(by_branch_cat)
# Compare Food vs total within each branch to judge the owner's suspicion:
food_share = (by_branch_cat / by_branch_cat.groupby("branch").transform("sum") * 100).round(1)
print(food_share)  # % of each branch's revenue per category

In [ ]:
# Q3: weekend share of revenue
weekend = df[df["weekday"].isin(["Saturday", "Sunday"])]
share = weekend["revenue"].sum() / df["revenue"].sum() * 100
print(f"Weekend share of revenue: {share:.1f}%")

In [ ]:
# Q4: products with low average rating + supporting chart
avg_rating = df.groupby("product")["customer_rating"].mean().sort_values()
print(avg_rating.round(2))
print("Below 3.5:", list(avg_rating[avg_rating < 3.5].index))

avg_rating.plot(kind="bar", title="Average customer rating per product")
plt.axhline(3.5, color="red", linestyle="--", label="Concern threshold (3.5)")
plt.ylabel("Average rating")
plt.legend()
plt.show()

---

🎉 **Well done!** You've completed the full data-analysis loop: load → inspect → clean → transform → aggregate → visualize → conclude. That loop is the same whether the dataset has 400 rows or 40 million.